# CAFA Notebook

This notebook is the only run interface for the project.

It should follow the actual project pipeline for the stages currently implemented. At the moment that means:

- resolve pinned official sources
- download and validate the GO ontology artifact
- load the ontology for downstream stages
- extract and validate the recreated train taxonomy artifact

Later stages such as train terms, train sequences, test extraction, benchmark construction, training, evaluation, and deployment will be added to this same notebook flow as they are implemented.

## Load dependencies

In [ ]:
from datetime import date
from pathlib import Path
from pprint import pprint
import shutil
import sys

PROJECT_ROOT = Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

## Load project modules

In [ ]:
from cafa import ProjectConfig
from cafa.io import build_recreated_layout, read_train_taxonomy_rows, write_train_taxonomy
from cafa.ontology import read_go_obo, subontology_terms
from cafa.sources import (
    download_and_validate_go_obo,
    download_source,
    extract_tar_gz_member,
    resolve_go_obo_snapshot,
    resolve_uniprot_sprot_snapshot,
    sha256_file,
)
from cafa.train import extract_train_taxonomy_records
from cafa.validation import validate_go_obo, validate_train_taxonomy

## Configure the current real pipeline run

In [ ]:
config = ProjectConfig(
    go_release="2025-06-01",
    train_uniprot_release="2025_03",
    submission_deadline=date(2026, 1, 27),
    evaluation_time=date(2026, 3, 17),
    train_taxon_ids=(
        9606, 10090, 3702, 559292, 10116, 284812, 83333, 7227,
        6239, 83332, 7955, 44689, 39947, 9913, 9031, 8355, 237561
    ),
    test_taxon_ids=(),
    subontologies=("MF", "BP", "CC"),
    evidence_codes=(
        "EXP", "IDA", "IPI", "IMP", "IGI", "IEP", "HTP",
        "HDA", "HMP", "HGI", "HEP", "TAS",  "IC",
    ),
    similarity_backend="diamond",
    validation_mode="canonical",
    project_root=PROJECT_ROOT,
    cache_dir=Path(".cache/cafa"),
    recreated_data_dir=Path("recreated_comp_data"),
    artifacts_dir=Path("artifacts"),
    results_dir=Path("results"),
)
layout = build_recreated_layout(PROJECT_ROOT / "recreated_comp_data")
pprint(
    {
        "train_taxon_ids": config.train_taxon_ids,
        "subontologies": config.subontologies,
        "evidence_codes": config.evidence_codes,
        "recreated_root": str(layout.root_dir),
    }
)

## Resolve pinned sources

In [ ]:
go_snapshot = resolve_go_obo_snapshot(config)
uniprot_snapshot = resolve_uniprot_sprot_snapshot(config)
pprint(
    {
        "go_snapshot": {
            "url": go_snapshot.url,
            "local_path": str(go_snapshot.local_path),
        },
        "uniprot_snapshot": {
            "url": uniprot_snapshot.url,
            "local_path": str(uniprot_snapshot.local_path),
        },
    }
)

## Recreate and validate `Train/go-basic.obo`

In [ ]:
go_reference_path = PROJECT_ROOT / "comp_data" / "Train" / "go-basic.obo"
downloaded_go_path = download_and_validate_go_obo(config, go_reference_path)
go_artifact_path = layout.train_dir / "go-basic.obo"
shutil.copyfile(downloaded_go_path, go_artifact_path)

go_report = validate_go_obo(go_artifact_path, go_reference_path)
if not go_report.passed:
    raise RuntimeError(go_report.message)

ontology = read_go_obo(go_artifact_path)
pprint(
    {
        "go_artifact_path": str(go_artifact_path),
        "go_sha256": sha256_file(go_artifact_path),
        "release": ontology.release,
        "num_terms": len(ontology.terms),
        "mf_terms": len(subontology_terms(ontology, "MF")),
        "bp_terms": len(subontology_terms(ontology, "BP")),
        "cc_terms": len(subontology_terms(ontology, "CC")),
        "validation_passed": go_report.passed,
    }
)

## Recreate and validate `Train/train_taxonomy.tsv`

In [ ]:
downloaded_uniprot_path = download_source(uniprot_snapshot)
extracted_uniprot_flatfile_path = extract_tar_gz_member(
    downloaded_uniprot_path,
    "uniprot_sprot.dat.gz",
)
train_taxonomy_rows = extract_train_taxonomy_records(config, uniprot_snapshot)
train_taxonomy_path = write_train_taxonomy(
    train_taxonomy_rows,
    layout.train_dir / "train_taxonomy.tsv",
)
train_taxonomy_report = validate_train_taxonomy(
    train_taxonomy_path,
    PROJECT_ROOT / "comp_data" / "Train" / "train_taxonomy.tsv",
    config,
)

In [ ]:
pprint(
    {
        "uniprot_archive_path": str(downloaded_uniprot_path),
        "uniprot_flatfile_gz_path": str(extracted_uniprot_flatfile_path),
        "train_taxonomy_path": str(train_taxonomy_path),
        "row_count": len(train_taxonomy_rows),
        "first_rows": train_taxonomy_rows[:3],
        "validation_passed": train_taxonomy_report.passed,
        "validation_message": train_taxonomy_report.message,
        "left_only_count": train_taxonomy_report.left_only_count,
        "right_only_count": train_taxonomy_report.right_only_count,
        "shared_mismatch_count": train_taxonomy_report.shared_mismatch_count,
        "left_path": train_taxonomy_report.left_path,
        "right_path": train_taxonomy_report.right_path,
        "validation_left_only_samples": train_taxonomy_report.sample_left_only,
        "validation_right_only_samples": train_taxonomy_report.sample_right_only,
    }
)

## Read back actual recreated artifacts

In [ ]:
actual_artifact_summary = {
    "go_artifact_exists": go_artifact_path.exists(),
    "train_taxonomy_artifact_exists": train_taxonomy_path.exists(),
    "train_taxonomy_preview": read_train_taxonomy_rows(train_taxonomy_path)[:5],
}
pprint(actual_artifact_summary)

## Current pipeline coverage

In [ ]:
pipeline_status = {
    "implemented_stages": {
        "Train/go-basic.obo": {
            "artifact_path": str(go_artifact_path),
            "validation_passed": go_report.passed,
            "left_only_count": go_report.left_only_count,
            "right_only_count": go_report.right_only_count,
            "shared_mismatch_count": go_report.shared_mismatch_count,
        },
        "Train/train_taxonomy.tsv": {
            "artifact_path": str(train_taxonomy_path),
            "validation_passed": train_taxonomy_report.passed,
            "validation_message": train_taxonomy_report.message,
            "left_only_count": train_taxonomy_report.left_only_count,
            "right_only_count": train_taxonomy_report.right_only_count,
            "shared_mismatch_count": train_taxonomy_report.shared_mismatch_count,
        },
    },
    "pending_stages": (
        "Extract and validate Train/train_terms.tsv",
        "Extract and validate Train/train_sequences.fasta",
        "Resolve RG4 and recreate the test artifacts",
        "Build the benchmark bundle",
        "Train and evaluate prediction models",
        "Optional deployment/inference workflow",
    ),
}
pprint(pipeline_status)